In [1]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
import random
import pandas as pd
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

torch.backends.cudnn.benchmark = True

Device: cuda


In [2]:


class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_base_transform = T.Compose([
    T.RandomResizedCrop(96, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
])

ssl_transform = TwoCropsTransform(ssl_base_transform)
supervised_transform = T.ToTensor()

In [3]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

In [4]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class BYOL(nn.Module):
    def __init__(self):
        super().__init__()
        self.online_encoder = Encoder()
        self.online_projector = MLP(512)
        self.online_predictor = MLP(256, 512, 256)

        self.target_encoder = Encoder()
        self.target_projector = MLP(512)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau=0.996):
        for online, target in zip(self.online_encoder.parameters(),
                                  self.target_encoder.parameters()):
            target.data = tau * target.data + (1 - tau) * online.data

    def forward(self, x1, x2):
        o1 = self.online_predictor(
            self.online_projector(self.online_encoder(x1))
        )

        with torch.no_grad():
            t2 = self.target_projector(
                self.target_encoder(x2)
            )

        loss = 2 - 2 * F.cosine_similarity(o1, t2.detach(), dim=-1)
        return loss.mean()

In [5]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = MLP(512)

    def forward(self, x):
        z = self.projector(self.encoder(x))
        return F.normalize(z, dim=1)


def nt_xent_loss(z1, z2, temperature=1.0):   # FIXED τ
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([
        torch.arange(N, 2*N),
        torch.arange(0, N)
    ]).to(z.device)

    return F.cross_entropy(sim, positives)

In [6]:
class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)
        return self.fc(feat)


def train_probe(encoder, train_loader, test_loader, epochs=10):
    model = LinearProbe(encoder).to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [7]:
tensor_aug = T.Compose([
    T.RandomResizedCrop(96, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
])

def jacobian_frobenius(model, x):
    x = x.clone().detach().to(device).requires_grad_(True)
    y = model(x)
    v = torch.randn_like(y)

    Jv = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=v
    )[0]

    return torch.sqrt(Jv.pow(2).sum() / x.size(0))


def compute_metrics(encoder, loader):
    encoder.eval()
    jac_vals, aug_vals, noise_vals = [], [], []

    for i, (x, _) in enumerate(loader):
        if i == 5:
            break

        x = x.to(device)

        jac_vals.append(jacobian_frobenius(encoder, x).item())

        x_aug = torch.stack([tensor_aug(img.cpu()) for img in x]).to(device)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        noise = torch.randn_like(x) * 0.1
        with torch.no_grad():
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return np.mean(jac_vals), np.mean(aug_vals), np.mean(noise_vals)

In [10]:

from torch.cuda.amp import autocast, GradScaler

lambda_jac = 1e-3
seeds = [0,1,2,3,4]

results_df = results_df = pd.read_csv("/kaggle/input/datasets/rajeshr1005/st110results/stl10_results_progress.csv")
print("Existing seeds:", results_df["Seed"].values)

for seed in seeds:

    print("\n====================================")
    print("JacReg | Seed:", seed)
    print("====================================")

    set_seed(seed)
    seed_start = time.time()

    train_dataset = torchvision.datasets.STL10(
        root="./data",
        split="train",
        download=True,
        transform=ssl_transform
    )

    supervised_train_dataset = torchvision.datasets.STL10(
        root="./data",
        split="train",
        download=False,
        transform=supervised_transform
    )

    test_dataset = torchvision.datasets.STL10(
        root="./data",
        split="test",
        download=False,
        transform=supervised_transform
    )

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    supervised_train_loader = DataLoader(supervised_train_dataset, batch_size=128, shuffle=True, num_workers=2)
    supervised_test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

    model = SimCLR().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    scaler = GradScaler()

    for epoch in range(100):

        epoch_start = time.time()
        model.train()

        for batch_idx, ((x1, x2), _) in enumerate(train_loader):

            x1, x2 = x1.to(device), x2.to(device)
            x1.requires_grad_(True)

            optimizer.zero_grad()

            with autocast():

                z1 = model(x1)
                z2 = model(x2)

                N = z1.size(0)
                z = torch.cat([z1, z2], dim=0).float()
                sim = torch.mm(z, z.t())
                mask = torch.eye(2*N, device=z.device).bool()
                sim.masked_fill_(mask, -1e4)

                positives = torch.cat([
                    torch.arange(N, 2*N),
                    torch.arange(0, N)
                ]).to(z.device)

                loss_ssl = F.cross_entropy(sim, positives)

            if batch_idx % 2 == 0:
                features = model.encoder(x1)
                v = torch.randn_like(features)
                Jv = torch.autograd.grad(
                    outputs=features,
                    inputs=x1,
                    grad_outputs=v,
                    retain_graph=True
                )[0]
                jac_reg = (Jv.pow(2).sum(dim=[1,2,3])).mean()
            else:
                jac_reg = 0.0

            loss = loss_ssl + lambda_jac * jac_reg

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        print(f"[JacReg] Epoch {epoch+1}/100 | {time.time()-epoch_start:.2f}s")

    encoder = model.encoder

    acc = train_probe(encoder, supervised_train_loader, supervised_test_loader)
    jac, aug, noise = compute_metrics(encoder, supervised_test_loader)

    seed_time = (time.time() - seed_start) / 60

    results_df.loc[results_df["Seed"] == seed, "SimCLR_JacReg_Jacobian"] = jac
    results_df.loc[results_df["Seed"] == seed, "SimCLR_JacReg_Aug"] = aug
    results_df.loc[results_df["Seed"] == seed, "SimCLR_JacReg_Noise"] = noise
    results_df.loc[results_df["Seed"] == seed, "SimCLR_JacReg_Acc"] = acc
    results_df.loc[results_df["Seed"] == seed, "SimCLR_JacReg_Time_Min"] = seed_time

    results_df.to_csv("/kaggle/working/stl10_results_progress.csv", index=False)

    print("Seed done in", seed_time, "minutes")

Existing seeds: [0 1 2 3 4]

JacReg | Seed: 0


100%|██████████| 2.64G/2.64G [06:29<00:00, 6.77MB/s] 
/tmp/ipykernel_55/486527846.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/486527846.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[JacReg] Epoch 1/100 | 29.22s
[JacReg] Epoch 2/100 | 11.45s
[JacReg] Epoch 3/100 | 11.28s
[JacReg] Epoch 4/100 | 11.24s
[JacReg] Epoch 5/100 | 11.45s
[JacReg] Epoch 6/100 | 11.45s
[JacReg] Epoch 7/100 | 11.26s
[JacReg] Epoch 8/100 | 11.46s
[JacReg] Epoch 9/100 | 11.57s
[JacReg] Epoch 10/100 | 11.31s
[JacReg] Epoch 11/100 | 11.10s
[JacReg] Epoch 12/100 | 11.16s
[JacReg] Epoch 13/100 | 11.16s
[JacReg] Epoch 14/100 | 11.75s
[JacReg] Epoch 15/100 | 11.71s
[JacReg] Epoch 16/100 | 11.70s
[JacReg] Epoch 17/100 | 11.58s
[JacReg] Epoch 18/100 | 11.28s
[JacReg] Epoch 19/100 | 11.67s
[JacReg] Epoch 20/100 | 11.49s
[JacReg] Epoch 21/100 | 11.71s
[JacReg] Epoch 22/100 | 11.54s
[JacReg] Epoch 23/100 | 11.23s
[JacReg] Epoch 24/100 | 11.10s
[JacReg] Epoch 25/100 | 11.18s
[JacReg] Epoch 26/100 | 11.76s
[JacReg] Epoch 27/100 | 11.69s
[JacReg] Epoch 28/100 | 11.36s
[JacReg] Epoch 29/100 | 11.39s
[JacReg] Epoch 30/100 | 11.28s
[JacReg] Epoch 31/100 | 11.55s
[JacReg] Epoch 32/100 | 11.33s
[JacReg] Epoch 33

/tmp/ipykernel_55/486527846.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/486527846.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[JacReg] Epoch 1/100 | 11.66s
[JacReg] Epoch 2/100 | 11.50s
[JacReg] Epoch 3/100 | 11.47s
[JacReg] Epoch 4/100 | 11.44s
[JacReg] Epoch 5/100 | 11.51s
[JacReg] Epoch 6/100 | 11.49s
[JacReg] Epoch 7/100 | 11.30s
[JacReg] Epoch 8/100 | 11.61s
[JacReg] Epoch 9/100 | 11.36s
[JacReg] Epoch 10/100 | 11.68s
[JacReg] Epoch 11/100 | 11.72s
[JacReg] Epoch 12/100 | 11.33s
[JacReg] Epoch 13/100 | 11.67s
[JacReg] Epoch 14/100 | 11.36s
[JacReg] Epoch 15/100 | 11.44s
[JacReg] Epoch 16/100 | 11.65s
[JacReg] Epoch 17/100 | 11.58s
[JacReg] Epoch 18/100 | 11.36s
[JacReg] Epoch 19/100 | 11.51s
[JacReg] Epoch 20/100 | 11.30s
[JacReg] Epoch 21/100 | 11.88s
[JacReg] Epoch 22/100 | 11.37s
[JacReg] Epoch 23/100 | 11.38s
[JacReg] Epoch 24/100 | 11.49s
[JacReg] Epoch 25/100 | 11.41s
[JacReg] Epoch 26/100 | 11.61s
[JacReg] Epoch 27/100 | 11.60s
[JacReg] Epoch 28/100 | 11.53s
[JacReg] Epoch 29/100 | 11.36s
[JacReg] Epoch 30/100 | 11.24s
[JacReg] Epoch 31/100 | 11.36s
[JacReg] Epoch 32/100 | 11.79s
[JacReg] Epoch 33

...

[JacReg] Epoch 1/100 | 11.56s
[JacReg] Epoch 2/100 | 11.47s
[JacReg] Epoch 3/100 | 11.50s
[JacReg] Epoch 4/100 | 11.44s
[JacReg] Epoch 5/100 | 11.47s
[JacReg] Epoch 6/100 | 11.45s
[JacReg] Epoch 7/100 | 11.75s
[JacReg] Epoch 8/100 | 11.61s
[JacReg] Epoch 9/100 | 11.96s
[JacReg] Epoch 10/100 | 11.48s
[JacReg] Epoch 11/100 | 11.28s
[JacReg] Epoch 12/100 | 11.26s
[JacReg] Epoch 13/100 | 11.65s
[JacReg] Epoch 14/100 | 11.58s
[JacReg] Epoch 15/100 | 11.63s
[JacReg] Epoch 16/100 | 11.48s
[JacReg] Epoch 17/100 | 11.60s
[JacReg] Epoch 18/100 | 11.56s
[JacReg] Epoch 19/100 | 11.10s
[JacReg] Epoch 20/100 | 11.46s
[JacReg] Epoch 21/100 | 11.08s
[JacReg] Epoch 22/100 | 11.33s
[JacReg] Epoch 23/100 | 11.25s
[JacReg] Epoch 24/100 | 11.25s
[JacReg] Epoch 25/100 | 11.41s
[JacReg] Epoch 26/100 | 11.52s
[JacReg] Epoch 27/100 | 11.24s
[JacReg] Epoch 28/100 | 11.56s
[JacReg] Epoch 29/100 | 11.39s
[JacReg] Epoch 30/100 | 11.57s
[JacReg] Epoch 31/100 | 11.39s
[JacReg] Epoch 32/100 | 11.30s
[JacReg] Epoch 33

/tmp/ipykernel_55/486527846.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/486527846.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[JacReg] Epoch 1/100 | 11.23s
[JacReg] Epoch 2/100 | 11.22s
[JacReg] Epoch 3/100 | 11.67s
[JacReg] Epoch 4/100 | 11.34s
[JacReg] Epoch 5/100 | 11.62s
[JacReg] Epoch 6/100 | 11.33s
[JacReg] Epoch 7/100 | 11.81s
[JacReg] Epoch 8/100 | 11.62s
[JacReg] Epoch 9/100 | 11.70s
[JacReg] Epoch 10/100 | 11.78s
[JacReg] Epoch 11/100 | 11.72s
[JacReg] Epoch 12/100 | 11.39s
[JacReg] Epoch 13/100 | 11.85s
[JacReg] Epoch 14/100 | 11.37s
[JacReg] Epoch 15/100 | 11.99s
[JacReg] Epoch 16/100 | 11.75s
[JacReg] Epoch 17/100 | 11.54s
[JacReg] Epoch 18/100 | 11.49s
[JacReg] Epoch 19/100 | 11.54s
[JacReg] Epoch 20/100 | 11.65s
[JacReg] Epoch 21/100 | 11.39s
[JacReg] Epoch 22/100 | 11.72s
[JacReg] Epoch 23/100 | 11.49s
[JacReg] Epoch 24/100 | 11.68s
[JacReg] Epoch 25/100 | 11.67s
[JacReg] Epoch 26/100 | 11.45s
[JacReg] Epoch 27/100 | 11.40s
[JacReg] Epoch 28/100 | 11.54s
[JacReg] Epoch 29/100 | 11.68s
[JacReg] Epoch 30/100 | 11.63s
[JacReg] Epoch 31/100 | 11.92s
[JacReg] Epoch 32/100 | 11.71s
[JacReg] Epoch 33

/tmp/ipykernel_55/486527846.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/486527846.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[JacReg] Epoch 1/100 | 11.26s
[JacReg] Epoch 2/100 | 11.08s
[JacReg] Epoch 3/100 | 11.49s
[JacReg] Epoch 4/100 | 11.31s
[JacReg] Epoch 5/100 | 11.35s
[JacReg] Epoch 6/100 | 11.32s
[JacReg] Epoch 7/100 | 11.22s
[JacReg] Epoch 8/100 | 11.77s
[JacReg] Epoch 9/100 | 11.43s
[JacReg] Epoch 10/100 | 11.69s
[JacReg] Epoch 11/100 | 11.87s
[JacReg] Epoch 12/100 | 11.26s
[JacReg] Epoch 13/100 | 11.18s
[JacReg] Epoch 14/100 | 11.77s
[JacReg] Epoch 15/100 | 11.56s
[JacReg] Epoch 16/100 | 11.44s
[JacReg] Epoch 17/100 | 11.46s
[JacReg] Epoch 18/100 | 11.84s
[JacReg] Epoch 19/100 | 11.47s
[JacReg] Epoch 20/100 | 11.48s
[JacReg] Epoch 21/100 | 11.33s
[JacReg] Epoch 22/100 | 11.25s
[JacReg] Epoch 23/100 | 11.57s
[JacReg] Epoch 24/100 | 11.57s
[JacReg] Epoch 25/100 | 11.62s
[JacReg] Epoch 26/100 | 11.65s
[JacReg] Epoch 27/100 | 10.97s
[JacReg] Epoch 28/100 | 11.14s
[JacReg] Epoch 29/100 | 11.36s
[JacReg] Epoch 30/100 | 11.48s
[JacReg] Epoch 31/100 | 11.27s
[JacReg] Epoch 32/100 | 11.38s
[JacReg] Epoch 33